In [ ]:
import os, subprocess, glob
def sh(cmd):
    print('$', cmd, flush=True); r = subprocess.run(cmd, shell=True, executable='/bin/bash'); print('exit', r.returncode, flush=True)
sh('pip install -q transformers 2>&1 | tail -1')
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModel
f = glob.glob('/kaggle/input/**/smiles.csv', recursive=True)[0]; smi = pd.read_csv(f).smiles.tolist(); print(len(smi), flush=True)
torch.set_num_threads(4); name = 'DeepChem/ChemBERTa-77M-MTR'; tok = AutoTokenizer.from_pretrained(name); mdl = AutoModel.from_pretrained(name).eval()
embs = []
for i in range(0, len(smi), 64):
    b = tok(smi[i:i+64], return_tensors='pt', padding=True, truncation=True, max_length=256)
    with torch.no_grad(): h = mdl(**b).last_hidden_state
    m = b['attention_mask'].unsqueeze(-1).float(); embs.append(((h * m).sum(1) / m.sum(1)).numpy())
    if (i // 64) % 20 == 0: print(i, flush=True)
os.makedirs('/kaggle/working/out', exist_ok=True); np.savez_compressed('/kaggle/working/out/herg_chemberta.npz', smiles=np.array(smi, dtype=object), E=np.vstack(embs).astype(np.float32))
print('NOTEBOOK_COMPLETED_OK')